In [ ]:
# 구글 드라이브 공유 링크로 파일 다운로드
!pip install -U gdown
!pip install -qqq datasets # huggingface's lib.
!pip install -qqq transformers==4.48.3
!pip install -qqq accelerate==0.28.0
!pip install tensorboard
!pip install -U accelerate

In [ ]:
import gdown
import zipfile
import os

# 파일 ID 입력
file_id = "14Yc5eDlrEvx4wWp6gtSiJ52ne_noEl9u"

# 다운로드 받을 파일 이름 지정
output = "test.zip"

# 다운로드 수행
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=True)

# 결과 파일 저장 경로
os.makedirs("release", exist_ok=True)

# 압축 해제할 경로
extract_dir = "data"
os.makedirs(extract_dir, exist_ok=True)

# 압축 풀기
with zipfile.ZipFile("test.zip", 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("압축 해제 완료:", extract_dir)

In [ ]:
# Task 3 에서 검색할 Top-k 유사 이미지 개수
# Task 3 진행시만 사용
TOP_K = 5

In [ ]:
# 추가 패키지 설치 시 '버전 정보' 꼭 명시하여 설치:
# !pip install name_of_package==X.X.X

# 버전 명시 안해주시면 TA가 테스트할 때 버전 충돌이 자주 발생합니다.
# 불이익 받지 않도록 버전 잘 명시해주시기 바랍니다.

In [ ]:
# ===== 설정 & 모델 다운로드 =====
STUDENT_ID = "202502204"
MODEL_FILE_ID = "1roBI4OaoCKlVJQDUGRKFsbB0_wSgRSrH"   # task1.pt
BATCH_SIZE = 64
IMG_SIZE = 224

MODEL_PATH = "task1.pt"
if not os.path.exists(MODEL_PATH):
    gdown.download(f"https://drive.google.com/uc?id={MODEL_FILE_ID}", MODEL_PATH, quiet=False)
print("ready:", MODEL_PATH)

In [ ]:
# --- 모델 정의 (학습 코드와 동일 구조) ---
import torch, torch.nn as nn
from torchvision import models, transforms

NUM_FRUIT, NUM_STYLE = 6, 3

class DualHeadClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.efficientnet_b0(weights=None)
        in_feats = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.2)
        self.fruit_head = nn.Linear(in_feats, NUM_FRUIT)
        self.style_head = nn.Linear(in_feats, NUM_STYLE)
    def forward(self, x):
        f = self.dropout(self.backbone(x))
        return self.fruit_head(f), self.style_head(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(MODEL_PATH, map_location=device)
state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
IMG_SIZE = ckpt.get("img_size", IMG_SIZE) if isinstance(ckpt, dict) else IMG_SIZE
model = DualHeadClassifier().to(device)
model.load_state_dict(state)
model.eval()

tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
print("loaded on", device, "| img_size", IMG_SIZE)

In [ ]:
# --- test 이미지 수집: data/ 하위에서 이미지 폴더 자동 탐색 (test 우선, id 숫자 정렬) ---
from glob import glob
EXTS = (".jpg", ".jpeg", ".png")

def find_image_dir(base="data"):
    cands = [r for r, _, fs in os.walk(base)
             if any(f.lower().endswith(EXTS) for f in fs)]
    for c in cands:                       # 'test' 경로 우선
        if "test" in c.replace("\\", "/").lower():
            return c
    return cands[0] if cands else base

IMG_DIR = find_image_dir("data")
files = [p for p in glob(os.path.join(IMG_DIR, "*")) if p.lower().endswith(EXTS)]
def id_key(p):
    stem = os.path.splitext(os.path.basename(p))[0]
    return (0, int(stem)) if stem.isdigit() else (1, stem)
files = sorted(set(files), key=id_key)
print("image dir:", IMG_DIR, "| images:", len(files))

In [ ]:
# --- 추론 & 출력 ---
from PIL import Image
os.makedirs("release", exist_ok=True)
out_path = f"release/{STUDENT_ID}.test.task1.txt"

@torch.no_grad()
def predict_batch(paths):
    imgs = torch.stack([tf(Image.open(p).convert("RGB")) for p in paths]).to(device)
    fp, sp = model(imgs)
    return fp.argmax(1).cpu().tolist(), sp.argmax(1).cpu().tolist()

with open(out_path, "w") as f:
    for i in range(0, len(files), BATCH_SIZE):
        batch = files[i:i + BATCH_SIZE]
        fr, st = predict_batch(batch)
        for path, fruit_label, style_label in zip(batch, fr, st):
            img_id = os.path.basename(path)
            print(f"{img_id}\t{style_label}\t{fruit_label}", file=f)
print("wrote", out_path)

In [ ]:
# --- 출력 확인 ---
with open(out_path) as f:
    lines = f.read().splitlines()
print("lines:", len(lines))
print("head:")
print("\n".join(lines[:5]))